
# 応用問題: MNIST 手書き数字の認識 (行列積 = AI推論)

## 目標

**ニューラルネットワークの推論 (forward) の正体は「行列積 + 活性化関数」** である。これまで何度も並列化してきた行列(ベクトル)積が, そのまま AI の推論計算になっていることを, **本物の MNIST 手書き数字**を認識して体感する。

題材は 2層 MLP:

- 入力 784次元 (28×28 画像を一列に並べたもの) -> 隠れ層 128 (ReLU) -> 出力 10クラス
- `h = ReLU(W1 x + b1)` (128次元), `o = W2 h + b2` (10次元), 予測クラス = `argmax(o)`

`data/` に **学習済みの重み**と **本物のテスト画像** が用意してある (いずれも NumPy 標準の **`.npy` 形式**: ヘッダ + 生バイナリ):

- `data/W1.npy, b1.npy, W2.npy, b2.npy` : 学習済みの重み (float64)
- `data/x_test.npy` : MNIST テスト画像 2000 枚 (各 784 画素, uint8 で 0..255)
- `data/y_test.npy` : 正解ラベル (int32, 0..9)

重みは別問題 **`02_mlp_train`** が MNIST を学習して書き出したものである (学習そのものは `02_mlp_train` で扱う)。テスト画像は世の中で配布されている MNIST (`mnist.npz`) から取り出したもの。`.npy` の読み込み関数 `read_npy` はソース内に用意済みなので, **入出力を書く必要はない** (並列化に集中せよ)。

## 課題

`mnist_infer.cpp` (または `mnist_infer.f90`) は, 重みと画像を読み込み, 各画像を forward して予測クラスを求め, 正解率を表示する。各画像の推論は互いに**独立**なので並列化できる。

`TODO` の箇所に **OpenMP の指示を追加**し, 画像のループを並列化せよ。

- C++: ループ直前に `#pragma omp parallel for reduction(+:correct)` を加える (正解数を数えるので reduction)。
- Fortran: ループを `!$omp parallel do private(...) reduction(+:correct)` と `!$omp end parallel do` で囲む。

各画像の推論で使う一時配列 (隠れ層 `h`) は各スレッドで別々に持つ必要がある (C++ はループ内で宣言済み, Fortran は `private` 指定)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer.exe

# Fortran
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer.exe
```

```
OMP_NUM_THREADS=4 ./mnist_infer.exe
```

## 期待される結果

```
MNIST テスト 2000 枚: 正解 1869 枚, 正解率 = 93.45%
```

- 学習済みの重みを使うので, **本当に手書き数字を 9 割以上当てる**。並列化した行列積がそのまま認識器になっている。
- 各画像の計算は独立かつ決定的なので, **スレッド数を変えても正解率は同じ** (93.45%) になる。
- スレッド数を増やすと速くなる (「性能を比べる」セルで台数効果を測れる)。
- (発展) GPU にオフロードする, 行列積を SIMD 化する, などにも挑戦してみよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mnist_infer.cpp
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <omp.h>

/* 本物の MNIST 手書き数字を, 学習済みの2層MLPで認識する (推論=forward)。
   重みは 02_mlp_train が学習して書き出したもの (784->128->10):
   - data/W1.npy, b1.npy, W2.npy, b2.npy : 学習済みの重み (float64)
   - data/x_test.npy : テスト画像 (uint8 [N,784], 画素 0..255)
   - data/y_test.npy : 正解ラベル (int32 [N], 0..9)
   推論の中身は「行列ベクトル積 + 活性化(ReLU) + argmax」。これまで並列化してきた
   行列計算が, そのまま手書き数字の認識になる。各画像の推論は独立なので並列化できる。

   入力はNumPy標準の .npy 形式 (ヘッダ + 生バイナリ)。read_npy は下に用意してある
   (I/O は本問題の主眼ではない)。 */

/* ---- .npy 読み込み: 任意の数値型を double 配列に読み込む (C順, 1〜2次元) ---- */
static double * read_npy(const char * path, long shape[2], int * ndim) {
  FILE * f = fopen(path, "rb");
  if (!f) { printf("%s が開けません\n", path); exit(1); }
  unsigned char magic[10];
  if (fread(magic, 1, 10, f) != 10 || memcmp(magic, "\x93NUMPY", 6) != 0) {
    printf("%s は .npy ではありません\n", path); exit(1);
  }
  int hlen = magic[8] | (magic[9] << 8);           /* ヘッダ(辞書文字列)の長さ */
  char * hdr = (char *)malloc(hlen + 1);
  fread(hdr, 1, hlen, f); hdr[hlen] = '\0';
  /* dtype (例: '<f8','|u1','<i4','<i8') と shape をヘッダ文字列から取り出す */
  char descr[8] = {0};
  { char * q = strstr(hdr, "descr"); q = strchr(q, ':'); q = strchr(q, '\'') + 1;
    int i = 0; while (*q != '\'' && i < 7) descr[i++] = *q++; descr[i] = '\0'; }
  long s0 = 1, s1 = 1; *ndim = 1;
  char * sp = strstr(hdr, "shape"); sp = strchr(sp, '(') + 1;
  s0 = atol(sp);
  char * comma = strchr(sp, ',');
  char * after = comma + 1; while (*after == ' ') after++;
  if (*after != ')') { s1 = atol(after); *ndim = 2; } else { s1 = 1; *ndim = 1; }
  shape[0] = s0; shape[1] = s1;
  long n = s0 * (*ndim == 2 ? s1 : 1);
  free(hdr);

  double * out = (double *)malloc(sizeof(double) * n);
  if (!strcmp(descr, "<f8")) {
    fread(out, sizeof(double), n, f);
  } else if (!strcmp(descr, "<f4")) {
    float * t = (float *)malloc(sizeof(float) * n); fread(t, sizeof(float), n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "|u1")) {
    unsigned char * t = (unsigned char *)malloc(n); fread(t, 1, n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "<i4")) {
    int * t = (int *)malloc(sizeof(int) * n); fread(t, sizeof(int), n, f);
    for (long i = 0; i < n; i++) out[i] = t[i]; free(t);
  } else if (!strcmp(descr, "<i8")) {
    long long * t = (long long *)malloc(sizeof(long long) * n); fread(t, sizeof(long long), n, f);
    for (long i = 0; i < n; i++) out[i] = (double)t[i]; free(t);
  } else {
    printf("%s: 未対応の dtype %s\n", path, descr); exit(1);
  }
  fclose(f);
  return out;
}

int main(int argc, char ** argv) {
  long sh[2]; int nd;

  /* --- 学習済みの重みの読み込み --- */
  double * W1 = read_npy("data/W1.npy", sh, &nd); int HID = sh[0], IN = sh[1];
  double * b1 = read_npy("data/b1.npy", sh, &nd);
  double * W2 = read_npy("data/W2.npy", sh, &nd); int OUT = sh[0];
  double * b2 = read_npy("data/b2.npy", sh, &nd);

  /* --- テスト画像の読み込み (画素 0..255 -> 0..1 に正規化) --- */
  double * Xu = read_npy("data/x_test.npy", sh, &nd); int NT = sh[0];
  double * yd = read_npy("data/y_test.npy", sh, &nd);
  double * X = (double *)malloc(sizeof(double) * (long)NT * IN);
  int    * y = (int *)malloc(sizeof(int) * NT);
  for (long i = 0; i < (long)NT * IN; i++) X[i] = Xu[i] / 255.0;
  for (int i = 0; i < NT; i++)             y[i] = (int)yd[i];
  free(Xu); free(yd);

  /* --- 推論: 各画像を MLP に通して予測クラス(argmax)を求め, 正解数を数える --- */
  long correct = 0;
  double t0 = omp_get_wtime();
  // TODO: 各画像の推論は独立。#pragma omp parallel for reduction(+:correct) で並列化せよ.
  for (int i = 0; i < NT; i++) {
    double h[1024];                       /* 隠れ層 (HID<=1024 を仮定) */
    const double * x = &X[(long)i * IN];
    for (int hh = 0; hh < HID; hh++) {    /* h = ReLU(W1 x + b1) */
      double s = b1[hh];
      const double * w = &W1[(long)hh * IN];
      for (int k = 0; k < IN; k++) s += w[k] * x[k];
      h[hh] = (s > 0.0) ? s : 0.0;
    }
    int best = 0; double bestv = -1e300;  /* o = W2 h + b2, argmax */
    for (int oo = 0; oo < OUT; oo++) {
      double s = b2[oo];
      const double * w = &W2[(long)oo * HID];
      for (int hh = 0; hh < HID; hh++) s += w[hh] * h[hh];
      if (s > bestv) { bestv = s; best = oo; }
    }
    if (best == y[i]) correct++;
  }
  double elapsed = omp_get_wtime() - t0;

  printf("MNIST テスト %d 枚: 正解 %ld 枚, 正解率 = %.2f%%\n",
         NT, correct, 100.0 * correct / NT);
  printf("elapsed = %.3f sec\n", elapsed);
  free(W1); free(b1); free(W2); free(b2); free(X); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mnist_infer.cpp -o mnist_infer_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mnist_infer.f90
! 本物の MNIST 手書き数字を, 学習済みの2層MLPで認識する (推論=forward)。
! 重みは 02_mlp_train が学習して書き出したもの (784->128->10):
!   data/W1.npy, b1.npy, W2.npy, b2.npy : 学習済みの重み (float64)
!   data/x_test.npy : テスト画像 (uint8 [N,784], 画素 0..255)
!   data/y_test.npy : 正解ラベル (int32 [N], 0..9)
! 推論の中身は「行列ベクトル積 + 活性化(ReLU) + argmax」。各画像の推論は独立なので並列化できる。
! 入力はNumPy標準の .npy 形式。read_npy は下のモジュールに用意してある (I/O は主眼ではない)。

module npy_io
contains
  ! .npy を読み, 中身を「保存順 (C順) のまま」flat な real(8) 配列 a(1:n) に入れる。
  ! 形は s0,s1 (1次元なら ndim=1, s1=1)。dtype は f8/f4/u1/i4/i8 に対応。
  subroutine read_npy(path, a, s0, s1, ndim)
    character(*), intent(in) :: path
    real(8), allocatable, intent(out) :: a(:)
    integer, intent(out) :: s0, s1, ndim
    integer :: u, hlen, p1, p2, ios
    integer(8) :: n
    character(len=10) :: magic
    character(len=:), allocatable :: hdr, sub
    character(len=8) :: descr
    integer(1), allocatable :: t1(:)
    integer(4), allocatable :: t4(:)
    integer(8), allocatable :: t8(:)
    real(4),    allocatable :: r4(:)
    integer(8) :: i

    open(newunit=u, file=path, access='stream', form='unformatted', &
         status='old', action='read')
    read(u) magic
    if (magic(1:6) /= char(147)//'NUMPY') stop '.npy ではありません'
    hlen = ichar(magic(9:9)) + 256*ichar(magic(10:10))
    allocate(character(len=hlen) :: hdr)
    read(u) hdr
    ! dtype
    if      (index(hdr,"'<f8'") > 0) then; descr = '<f8'
    else if (index(hdr,"'<f4'") > 0) then; descr = '<f4'
    else if (index(hdr,"'|u1'") > 0) then; descr = '|u1'
    else if (index(hdr,"'<i4'") > 0) then; descr = '<i4'
    else if (index(hdr,"'<i8'") > 0) then; descr = '<i8'
    else; stop '未対応の dtype'; end if
    ! shape: '(' と ')' の間を取り出し, カンマを空白にして読む
    p1 = index(hdr, '('); p2 = index(hdr, ')')
    sub = hdr(p1+1:p2-1)
    do i = 1, len(sub); if (sub(i:i) == ',') sub(i:i) = ' '; end do
    read(sub, *, iostat=ios) s0, s1
    if (ios /= 0) then; ndim = 1; s1 = 1; read(sub, *) s0
    else;               ndim = 2; end if
    n = int(s0,8) * merge(int(s1,8), 1_8, ndim == 2)

    allocate(a(n))
    select case (trim(descr))
    case ('<f8'); read(u) a
    case ('<f4'); allocate(r4(n)); read(u) r4; a = real(r4,8)
    case ('|u1'); allocate(t1(n)); read(u) t1
                  do i = 1, n; a(i) = real(merge(int(t1(i))+256, int(t1(i)), t1(i) < 0), 8); end do
    case ('<i4'); allocate(t4(n)); read(u) t4; a = real(t4,8)
    case ('<i8'); allocate(t8(n)); read(u) t8; a = real(t8,8)
    end select
    close(u)
  end subroutine read_npy
end module npy_io

program mnist_infer
  use npy_io
  use omp_lib
  implicit none
  integer :: IN, HID, OUT, NT, i, hh, oo, k, best, s0, s1, nd
  integer(8) :: correct
  real(8) :: s, bestv, t0, elapsed
  real(8), allocatable :: W1(:,:), b1(:), W2(:,:), b2(:), X(:,:), hidv(:)
  real(8), allocatable :: flat(:)
  integer, allocatable :: y(:)

  ! --- 重みの読み込み (.npy は C順なので reshape で Fortran の (k,hh) 並びになる) ---
  call read_npy("data/W1.npy", flat, s0, s1, nd); HID = s0; IN = s1
  W1 = reshape(flat, [IN, HID])              ! W1(k,hh) = W1[hh][k]
  call read_npy("data/b1.npy", flat, s0, s1, nd); b1 = flat
  call read_npy("data/W2.npy", flat, s0, s1, nd); OUT = s0
  W2 = reshape(flat, [HID, OUT])             ! W2(hh,oo) = W2[oo][hh]
  call read_npy("data/b2.npy", flat, s0, s1, nd); b2 = flat

  ! --- テスト画像の読み込み (画素 0..255 -> 0..1) ---
  call read_npy("data/x_test.npy", flat, s0, s1, nd); NT = s0
  X = reshape(flat, [IN, NT]) / 255.0d0      ! X(k,i) = 画像 i の画素 k
  call read_npy("data/y_test.npy", flat, s0, s1, nd)
  allocate(y(NT)); y = nint(flat)

  ! --- 推論 ---
  correct = 0
  t0 = omp_get_wtime()
  ! TODO: 各画像の推論は独立。!$omp parallel do reduction(+:correct) で並列化せよ.
  do i = 1, NT
     allocate(hidv(HID))
     do hh = 1, HID                       ! h = ReLU(W1 x + b1)
        s = b1(hh)
        do k = 1, IN
           s = s + W1(k,hh) * X(k,i)
        end do
        hidv(hh) = max(0.0d0, s)
     end do
     best = 1; bestv = -1d300              ! o = W2 h + b2, argmax
     do oo = 1, OUT
        s = b2(oo)
        do hh = 1, HID
           s = s + W2(hh,oo) * hidv(hh)
        end do
        if (s > bestv) then
           bestv = s; best = oo
        end if
     end do
     if (best - 1 == y(i)) correct = correct + 1   ! クラスは 0..9, best は 1..10
     deallocate(hidv)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,f0.2,a)", &
       "MNIST テスト ", NT, " 枚: 正解 ", correct, " 枚, 正解率 = ", &
       100.0d0 * correct / NT, " %"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program mnist_infer

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mnist_infer.f90 -o mnist_infer_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mnist_infer_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mnist_infer.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mnist_infer.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mnist_infer.cpp}